# 01 - Exploration & Basic Statistics (Task 2)
**Owner:** Asjad Aleem

Data: balanced sample from `src/load_data.py` (raw dataset has 1,038,340 rows).

In [ ]:
import sys, re
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
df = pd.read_csv(ROOT / "data/processed/sample_raw.csv")
print(df.shape)

## 1. Basic statistics

In [ ]:
df["doc"] = df["Headline"].fillna("") + " " + df["News Text"].fillna("")
df["n_words"] = df["doc"].str.split().str.len()
print("Number of documents :", len(df))
print("Avg length (words)  :", round(df["n_words"].mean(), 1))
print("Median length       :", df["n_words"].median())
print("Min / Max length    :", df["n_words"].min(), "/", df["n_words"].max())
df["n_words"].describe()

## 2. Target distribution
Sample is balanced on purpose (same number per category) so no class dominates later models.

In [ ]:
counts = df["Category"].value_counts()
print(counts)
ax = counts.plot(kind="bar", color="#4C72B0", title="Articles per category")
ax.set_ylabel("articles"); plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(ROOT / "figures/category_distribution.png", dpi=150); plt.show()

In [ ]:
print(df["Source"].value_counts().head(10))
ax = df["n_words"].clip(upper=2000).plot(kind="hist", bins=50, title="Document length (words, clipped at 2000)")
ax.set_xlabel("words"); plt.tight_layout(); plt.savefig(ROOT / "figures/doc_length_hist.png", dpi=150); plt.show()

## 3. Tokens vs types (on a sample of 1,000 documents)

In [ ]:
sample = df.sample(n=min(1000, len(df)), random_state=42)
tokens = [t for d in sample["doc"] for t in d.split()]
freq = Counter(tokens)
print("Tokens (total words) :", f"{len(tokens):,}")
print("Types  (unique words):", f"{len(freq):,}")
print("Type/Token ratio     :", round(len(freq) / len(tokens), 4))
hapax = sum(1 for c in freq.values() if c == 1)
print("Words seen only once :", f"{hapax:,}", f"({100*hapax/len(freq):.1f}% of vocab)")
freq.most_common(20)

## 4. Zipf's law / long tail
If Zipf holds, rank vs frequency on a log-log plot is close to a straight line.

In [ ]:
import numpy as np
f = np.array(sorted(freq.values(), reverse=True)); r = np.arange(1, len(f) + 1)
plt.figure(figsize=(6, 4))
plt.loglog(r, f, marker=".", linestyle="none", markersize=2, label="our data")
plt.loglog(r, f[0] / r, "r--", label="ideal Zipf (1/rank)")
plt.xlabel("rank (log)"); plt.ylabel("frequency (log)"); plt.title("Zipf plot"); plt.legend()
plt.tight_layout(); plt.savefig(ROOT / "figures/zipf.png", dpi=150); plt.show()
top20 = sum(c for _, c in freq.most_common(20))
print(f"Top 20 words cover {100*top20/len(tokens):.1f}% of all tokens")

**Comment (edit in your own words):** The top words are grammar words like کے، کی، میں، سے، ہے.
Only 20 words cover a big share of all tokens, while a large part of the vocabulary appears only once (the long tail).
The log-log plot is close to a straight line, so the data follows Zipf's law.

## 5. Messy text examples
The authors already removed punctuation, English and diacritics. But that cleaning **damaged** the text in a few ways:

In [ ]:
def show(pattern, name, n=3, width=60):
    rx = re.compile(pattern)
    hits = df["News Text"].str.count(pattern).sum()
    print(f"=== {name}: {hits:,} matches ===")
    for t in df["News Text"]:
        m = rx.search(t)
        if m:
            s = max(0, m.start() - width); print("   ...", t[s:m.end() + width], "...")
            n -= 1
            if n == 0: break
    print()

# (a) letter 'آ' was deleted: "اسلام آباد" -> "اسلام باد", "آئی ایم ایف" -> "ائی ایم ایف"
show(r"اسلام باد|\bائی\b", "Missing letter آ")
print("Count of آ in the whole sample:", df["News Text"].str.count("آ").sum())
# (b) punctuation removed -> sentences glued: "کرے گا۔ اس" -> "کرے گااس"
show(r"\b(?:گا|گی|گے|ہے|ہیں|تھا|تھی|تھے)(?:اس|ان|یہ|وہ)\w*", "Sentences glued together")
# (c) decimal point removed: "7.67 فیصد" -> "767 فیصد"
show(r"\b\d{3,}\s*فیصد", "Percent > 100 (lost decimal point)")

Other things to mention: very long articles (max length above), and many numbers in Business news.